# Human Gene Essentiality V2 - FINAL (No Leakage)
## Strict train/test separation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded!")

In [ ]:
DATA_DIR = '/content/drive/MyDrive/depmap_data'
print("Files:")
for f in os.listdir(DATA_DIR):
    print(f"  {f}")

In [ ]:
# Load CRISPR
print("Loading CRISPR...")
crispr_raw = pd.read_csv(os.path.join(DATA_DIR, 'CRISPRGeneEffect.csv'), index_col=0)
print(f"Shape: {crispr_raw.shape} (cells x genes)")

In [ ]:
# Load Expression
print("\nLoading Expression...")
expr_file = [f for f in os.listdir(DATA_DIR) if 'expression' in f.lower() and 'crispr' not in f.lower()][0]
print(f"File: {expr_file}")

expr_full = pd.read_csv(os.path.join(DATA_DIR, expr_file))
print(f"Raw shape: {expr_full.shape}")

In [ ]:
# Find ModelID column and gene start
model_col = None
for col in expr_full.columns:
    try:
        if str(expr_full[col].iloc[0]).startswith('ACH-'):
            model_col = col
            print(f"ModelID column: '{col}'")
            break
    except: pass

gene_start = 0
for i, col in enumerate(expr_full.columns):
    if ' (' in str(col) and ')' in str(col):
        gene_start = i
        print(f"Gene columns start at {i}")
        break

expr_clean = expr_full.iloc[:, gene_start:].copy()
expr_clean.index = expr_full[model_col].values
print(f"Expression cleaned: {expr_clean.shape}")

In [ ]:
# Find common cells and genes
common_cells = sorted(list(set(crispr_raw.index) & set(expr_clean.index)))
print(f"Common cells: {len(common_cells)}")

def get_symbol(name):
    if ' (' in str(name):
        return str(name).split(' (')[0].upper()
    return str(name).upper()

crispr_sym = {get_symbol(c): c for c in crispr_raw.columns}
expr_sym = {get_symbol(c): c for c in expr_clean.columns}
common_genes = sorted(list(set(crispr_sym.keys()) & set(expr_sym.keys())))
print(f"Common genes: {len(common_genes)}")

In [ ]:
# Create aligned dataframes
crispr_cols = [crispr_sym[g] for g in common_genes]
expr_cols = [expr_sym[g] for g in common_genes]

crispr_sub = crispr_raw.loc[common_cells, crispr_cols].copy()
expr_sub = expr_clean.loc[common_cells, expr_cols].copy()

crispr_sub.columns = common_genes
expr_sub.columns = common_genes

# Transpose: genes x cells
crispr_df = crispr_sub.T
expr_df = expr_sub.T

print(f"CRISPR: {crispr_df.shape}")
print(f"Expression: {expr_df.shape}")

assert crispr_df.shape == expr_df.shape
print("✅ Aligned!")

In [ ]:
# Binarize
binary_mat = (crispr_df < -0.5).astype(int)
print(f"Binary: {binary_mat.shape}")

In [ ]:
# ========================================
# STRICT TRAIN/TEST SPLIT - NO LEAKAGE
# ========================================
all_cells = list(binary_mat.columns)
np.random.seed(42)
np.random.shuffle(all_cells)

N_TRAIN = 200
N_TEST = 100

TRAIN_CELLS = all_cells[:N_TRAIN]
TEST_CELLS = all_cells[N_TRAIN:N_TRAIN+N_TEST]

print(f"Train cells: {len(TRAIN_CELLS)}")
print(f"Test cells: {len(TEST_CELLS)}")
print(f"Overlap: {len(set(TRAIN_CELLS) & set(TEST_CELLS))}")

In [ ]:
# Gene classification using ONLY TRAIN cells
train_binary = binary_mat[TRAIN_CELLS]
train_expr = expr_df[TRAIN_CELLS]

gene_consensus_train = train_binary.mean(axis=1)

print(f"Pan-essential (≥90%): {(gene_consensus_train >= 0.9).sum()}")
print(f"Context-dep (10-90%): {((gene_consensus_train > 0.1) & (gene_consensus_train < 0.9)).sum()}")
print(f"Non-essential (≤10%): {(gene_consensus_train <= 0.1).sum()}")

In [ ]:
# V2 Predictor - NO LEAKAGE VERSION
class HedgeFundV2_NoLeak:
    """
    All features computed ONLY from training cells.
    Test cell's own data only used for its own expression value.
    """
    def __init__(self):
        self.ml = GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1)
        self.scaler = StandardScaler()
        self.fitted = False
        self.train_cells = None
        
    def _feat(self, bmat, emat, gene, cell, reference_cells):
        """
        Features computed using ONLY reference_cells (training cells).
        - consensus: from reference_cells only
        - expression stats: from reference_cells only
        - cell's own expression: allowed (it's the input)
        """
        # Consensus from reference cells only
        ref_cells_excl = [c for c in reference_cells if c != cell]
        cons = bmat.loc[gene, ref_cells_excl].mean()
        var = bmat.loc[gene, ref_cells_excl].var()
        
        # Expression stats from reference cells only
        expr_ref = emat.loc[gene, ref_cells_excl]
        expr_m = expr_ref.mean()
        expr_s = expr_ref.std() + 1e-6
        
        # Cell's own expression (this is the input, not leakage)
        expr = emat.loc[gene, cell]
        expr_z = (expr - expr_m) / expr_s
        
        # Percentile computed against reference cells only
        expr_p = (expr_ref < expr).mean()
        
        return [cons, var, expr, expr_z, expr_p, cons*expr, cons**2, expr**2]
    
    def fit(self, bmat, emat, train_cells):
        print("Training V2 (no leakage)...")
        self.train_cells = train_cells
        
        # Consensus from TRAIN cells only
        train_bmat = bmat[train_cells]
        cons = train_bmat.mean(axis=1)
        ctx_genes = cons[(cons > 0.1) & (cons < 0.9)].index.tolist()
        print(f"  Context genes: {len(ctx_genes)}")
        
        X, y = [], []
        
        # Leave-one-out within training set
        for c in tqdm(train_cells, desc="  Building features"):
            # Reference = all train cells except current
            ref = [x for x in train_cells if x != c]
            for g in ctx_genes:
                try:
                    f = self._feat(bmat, emat, g, c, ref)
                    if not np.isnan(f).any():
                        X.append(f)
                        y.append(bmat.loc[g, c])
                except: pass
        
        X, y = np.array(X), np.array(y)
        print(f"  Samples: {len(X):,}, Pos rate: {y.mean():.1%}")
        
        Xs = self.scaler.fit_transform(X)
        self.ml.fit(Xs, y)
        
        yp = self.ml.predict(Xs)
        print(f"  Train acc: {accuracy_score(y, yp):.3f}")
        print(f"  Train bal: {balanced_accuracy_score(y, yp):.3f}")
        
        names = ['cons', 'var', 'expr', 'expr_z', 'expr_p', 'c*e', 'c^2', 'e^2']
        imp = self.ml.feature_importances_
        print("  Feature importance:")
        for n, i in sorted(zip(names, imp), key=lambda x: -x[1]):
            print(f"    {n:10s}: {i:.3f}")
        
        self.fitted = True
        return self
    
    def predict(self, bmat, emat, cell):
        """
        Predict for a TEST cell using ONLY train cells as reference.
        """
        # Consensus from TRAIN cells only (no test cell info)
        cons = bmat[self.train_cells].mean(axis=1)
        
        preds = pd.Series(0, index=bmat.index)
        preds[cons >= 0.9] = 1  # Pan-essential
        
        # Context genes
        ctx = (cons > 0.1) & (cons < 0.9)
        ctx_genes = preds[ctx].index.tolist()
        
        if ctx_genes and self.fitted:
            X, valid = [], []
            for g in ctx_genes:
                try:
                    # Reference = ALL train cells (test cell not included)
                    f = self._feat(bmat, emat, g, cell, self.train_cells)
                    if not np.isnan(f).any():
                        X.append(f)
                        valid.append(g)
                except: pass
            if X:
                Xs = self.scaler.transform(np.array(X))
                for g, p in zip(valid, self.ml.predict(Xs)):
                    preds[g] = p
        return preds

print("HedgeFundV2_NoLeak defined!")

In [ ]:
# Train on TRAIN cells only
pred = HedgeFundV2_NoLeak()
pred.fit(binary_mat, expr_df, TRAIN_CELLS)

In [ ]:
# Evaluate on TEST cells (never seen during training)
yt, yp = [], []
for c in tqdm(TEST_CELLS, desc="Testing"):
    yt.extend(binary_mat[c].values)
    yp.extend(pred.predict(binary_mat, expr_df, c).values)

yt, yp = np.array(yt), np.array(yp)

print(f"\n{'='*60}")
print(f"V2 RESULTS (NO LEAKAGE)")
print(f"{'='*60}")
print(f"Test cells: {len(TEST_CELLS)}")
print(f"Accuracy:   {accuracy_score(yt, yp):.3f}")
print(f"Balanced:   {balanced_accuracy_score(yt, yp):.3f}")
print(f"F1:         {f1_score(yt, yp):.3f}")

In [ ]:
# By category (using TRAIN consensus for categorization)
v1 = {'Pan-essential (≥90%)': 0.982, 'Common (50-90%)': 0.434, 
      'Context-dep (10-50%)': 0.439, 'Rarely ess (0-10%)': 0.991, 'Non-essential (0%)': 1.0}

# Use TRAIN consensus for gene categories
gc = gene_consensus_train

cats = {
    'Pan-essential (≥90%)': gc >= 0.9,
    'Common (50-90%)': (gc >= 0.5) & (gc < 0.9),
    'Context-dep (10-50%)': (gc >= 0.1) & (gc < 0.5),
    'Rarely ess (0-10%)': (gc > 0) & (gc < 0.1),
    'Non-essential (0%)': gc == 0
}

print("\n" + "="*70)
print("V1 → V2 COMPARISON (NO LEAKAGE)")
print("="*70)

N = len(TEST_CELLS)
v2_results = {}
for name, mask in cats.items():
    n = mask.sum()
    if n > 0:
        m = np.tile(mask.values, N)
        v2_acc = accuracy_score(yt[m], yp[m])
        v1_acc = v1[name]
        d = v2_acc - v1_acc
        e = "✅" if d > 0.05 else "➡️" if d > -0.05 else "❌"
        v2_results[name] = v2_acc
        print(f"{name:25s}: {v1_acc:.1%} → {v2_acc:.1%} ({d:+.1%}) {e} (n={n})")

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(cats))
w = 0.35

v1_vals = [v1[c] for c in cats]
v2_vals = [v2_results.get(c, 0) for c in cats]

bars1 = ax.bar(x - w/2, v1_vals, w, label='V1 (Consensus)', color='#ff6b6b')
bars2 = ax.bar(x + w/2, v2_vals, w, label='V2 (+Expression, No Leak)', color='#4ecdc4')
ax.axhline(0.5, color='gray', ls='--', alpha=0.5, label='Random')
ax.set_xticks(x)
ax.set_xticklabels(cats.keys(), rotation=45, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('V1 vs V2: Accuracy by Gene Category (No Leakage)')
ax.legend()
ax.set_ylim(0, 1.1)

for b, v in zip(bars1, v1_vals):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02, f'{v:.0%}', ha='center', fontsize=9)
for b, v in zip(bars2, v2_vals):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02, f'{v:.0%}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()